Runs as is on rapidsai/notebooks:24.10a-cuda12.5-py3.10 

In [ ]:
%pip install polars hdbscan azure-storage-blob

In [ ]:
import cuml
import cupy as cp
from cuml.cluster.hdbscan import HDBSCAN

In [ ]:
%env AZURE_STORAGE_ACCOUNT_KEY=

In [ ]:
import numpy as np
from azure.storage.blob import BlobServiceClient
import io
import os

# Replace with your actual connection string
account_name="enclaveid"
account_key=os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
connection_string = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
container_name = 'enclaveid-dagster-prod-bucket'
blob_name = 'interests_embeddings/cm0i27jdj0000aqpa73ghpcxf.snappy'

# Create the BlobServiceClient object
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create the BlobClient object
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)

# Download the blob content to a bytes buffer
downloaded_blob = blob_client.download_blob()

# Save the blob data to a file
with open("data.snappy", "wb") as file:
    downloaded_blob.readinto(file)

import polars as pl
df = pl.read_parquet("data.snappy")

In [ ]:
embeddings_gpu = cp.asarray(df["embeddings"].to_numpy())

# Reduce the embeddings dimensions
umap_model = cuml.UMAP(
    n_neighbors=15, n_components=100, min_dist=0.1, metric="cosine"
)
reduced_data_gpu = umap_model.fit_transform(embeddings_gpu)

# Clustering for single interests
fine_cluster_labels = HDBSCAN(
    min_cluster_size=10,
    gen_min_span_tree=True,
    metric="euclidean",
).fit_predict(reduced_data_gpu.astype(np.float64).get())

fine_cluster_stats = np.unique(fine_cluster_labels, return_counts=True)

In [ ]:
coarse_cluster_labels = HDBSCAN(
    min_cluster_size=10,
    gen_min_span_tree=True,
    metric="euclidean",
    # By specifying an epsilon we coalesce similar clusters
    # into a single cluster representing a broader category
    cluster_selection_epsilon=0.3,
).fit_predict(reduced_data_gpu.astype(np.float64).get())

coarse_cluster_stats = np.unique(coarse_cluster_labels, return_counts=True)

In [ ]:
len(coarse_cluster_stats[0])